# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. All references to dataset entities use their unique `@id` fields, ensuring clarity and reproducibility.

### Dataset Source
Dataset Croissant schema URL: [https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# You may need matplotlib or seaborn for EDA later
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets and their fields using `@id` values.

In [ ]:
# List available record sets
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}, description: {getattr(rs, 'description', '')}")
    # List fields inside each record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', '')}")
    print()

## 3. Data Extraction
Load data from each record set into pandas DataFrames. Use record set and field `@id`s from the overview above.

In [ ]:
# Extract data for each record set using their @id
dataframes = {}
# Store record set IDs for easy reference
record_set_ids = [rs.id for rs in dataset.record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"DataFrame for Record Set @id: {rs_id} ({len(df)} rows)")
    print(df.columns.tolist())
    print(df.head())
    print()

# For illustration, let's select the first record set for further analysis
main_rs_id = record_set_ids[0]
df_main = dataframes[main_rs_id]
print(f"Working with Record Set @id: {main_rs_id}")
print(df_main.head())

## 4. Exploratory Data Analysis (EDA)
Apply processing steps: filter records, normalize numeric fields, categorize/group as appropriate. All fields referenced using their `@id`.

In [ ]:
# Let's identify numeric fields from the record set's fields
rs = dataset.record_set(main_rs_id)
numeric_fields = [f.id for f in rs.fields if getattr(f, 'data_type', '') in ['Float', 'Integer', 'Number']]
print("Numeric field @ids in main record set:")
print(numeric_fields)

# Assume the first numeric field is of interest for filtering
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
    # Example threshold for filtering
    threshold = 10
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field if available
    cat_fields = [f.id for f in rs.fields if getattr(f, 'data_type', '') in ['Text', 'Boolean']]
    if cat_fields:
        group_field_id = cat_fields[0]
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print("Grouped records:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields using their `@id`. Here, example visualizations show distributions and groupings for numeric and categorical fields.

In [ ]:
# Visualize the numeric field distribution
if numeric_fields:
    plt.figure(figsize=(6, 4))
    sns.histplot(df_main[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, show a boxplot
    if cat_fields:
        plt.figure(figsize=(6, 4))
        sns.boxplot(x=df_main[cat_fields[0]], y=df_main[numeric_field_id])
        plt.title(f"{numeric_field_id} by {cat_fields[0]}")
        plt.xlabel(cat_fields[0])
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
We explored the FAIR² dataset using `mlcroissant`. Using unique `@id`s for record sets and fields, we:
- Loaded metadata and examined available record sets
- Loaded tabular data into pandas DataFrames
- Filtered, normalized, and grouped records using field `@id`s
- Visualized data distributions and relationships

The dataset provides detailed, structured clinical and genetic insights into NC-21OHD-associated infertility, but is limited in sample size (N=3) and scope. For robust machine learning analyses, additional data would be required. For clinical and academic illustration, this FAIR² dataset is valuable in demonstrating genotype-phenotype correlation analysis.